# EXP-126 · 모델별 확신도 보정(temperature scaling) 후 로그 평균 (CPU)

**목적**: Gemma 제로샷은 평균 최대확률 0.996으로 틀릴 때도 거의 100% 확신함 → 평균에서 다른 모델을 덮어버림.
각 모델의 확률을 온도 T로 보정(valid 정답 로그손실 최소화)한 뒤 로그 확률을 평균.

- 조합: EXP-111_equal_mean_gemma2x(LB 0.96782)와 같은 구성 = (0.52·3B + 0.48·7B) · 8B · (Gemma 560 + 280)/2 각 1/3
- 5-fold 교차검증(20회 반복): 보정 전 647.0 → 보정·로그 평균 647.95 (차이는 우연 범위)
- 가중치 대신 모델별 T 한 개씩만 추정

In [ ]:
import numpy as np, pandas as pd
from scipy.optimize import minimize_scalar
from google.colab import drive
drive.mount("/content/drive")
SHARE_DIR = "/content/drive/MyDrive/ssafy/ai-challenge"
E, S = f"{SHARE_DIR}/experiment_data", f"{SHARE_DIR}/submissions"
P = ["prob_a","prob_b","prob_c","prob_d"]
F = {"3B":("EXP-007_train6042_resolution1280_valid_seed43.csv","EXP-007_train6042_resolution1280_test_probabilities_seed43.csv"),
     "7B":("EXP-034_qwen7b_language_only_train1600_resolution1280_valid_seed43.csv","EXP-040_qwen7b_train1600_resolution1280_test_probabilities_seed43.csv"),
     "8B":("EXP-102_qwen3-vl-8b-instruct_lora_r16_valid_probs.csv","EXP-102_qwen3-vl-8b-instruct_lora_r16_test_probs.csv"),
     "G4":("EXP-120_gemma4-31b_zeroshot_valid_probs.csv","EXP-120_gemma4-31b_zeroshot_test_probs.csv"),
     "G4b":("EXP-122_gemma4-31b_t280_short_valid_probs.csv","EXP-122_gemma4-31b_t280_short_test_probs.csv")}
valid = pd.read_csv(f"{SHARE_DIR}/split/valid_split_seed43.csv"); v_ids = valid.id.values
ans = valid.answer.map(dict(zip("abcd",range(4)))).values
REF = pd.read_csv(f"{S}/EXP-111_equal_mean_gemma2x.csv"); t_ids = REF.id.values
V = {k: pd.read_csv(f"{E}/{a}").set_index("id").loc[v_ids,P].values.astype(float) for k,(a,b) in F.items()}
T = {k: pd.read_csv(f"{E}/{b}").set_index("id").loc[t_ids,P].values.astype(float) for k,(a,b) in F.items()}

def temp(p, t):
    l = np.log(np.clip(p,1e-12,1))/t; l -= l.max(1, keepdims=True); e = np.exp(l); return e/e.sum(1, keepdims=True)
nll = lambda p: -np.log(np.clip(p[np.arange(len(p)), ans], 1e-12, 1)).mean()
Ts = {k: minimize_scalar(lambda t: nll(temp(V[k], t)), bounds=(0.3, 20), method="bounded").x for k in V}
for k in V: print(f"{k:4s} T={Ts[k]:.2f}  평균 최대확률 {V[k].max(1).mean():.3f}")

L = lambda D: {k: np.log(np.clip(temp(v, Ts[k]), 1e-12, 1)) for k, v in D.items()}
mix = lambda D: (0.52*D["3B"] + 0.48*D["7B"] + D["8B"] + (D["G4"] + D["G4b"])/2) / 3
print("보정 전 valid", (mix(V).argmax(1)==ans).sum(), "| 보정·로그 평균 valid", (mix(L(V)).argmax(1)==ans).sum())
sub = pd.DataFrame({"id": t_ids, "answer": ["abcd"[i] for i in mix(L(T)).argmax(1)]})
sub.to_csv(f"{S}/EXP-126_gemma2x_tempcal_logavg.csv", index=False)
print("0.96782 제출 대비 바뀐 test:", (sub.answer.values != REF.answer.values).sum())